In [ ]:
from pathlib import Path
import pandas as pd
import sys
# Resolve project root directory
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.scaling import (
    MODEL_FEATURES,
    MISSINGNESS_FEATURE,
    LSTM_FEATURES,
    scale_train_validation_test,
    save_scaler,
)

In [ ]:
TRAIN_PATH = Path(
    "/home/jetson/Desktop/ciesol-anomaly-detection/data/processed/train_features_imputed_NaN_before_LSTM.parquet"
)

if not TRAIN_PATH.exists():
    raise FileNotFoundError(
        f"Train file not found: {TRAIN_PATH}"
    )

df_train = pd.read_parquet(TRAIN_PATH)
TEST_PATH = Path(
    "/home/jetson/Desktop/ciesol-anomaly-detection/data/processed/test_features_imputed_NaN_before_LSTM.parquet"
)

if not TEST_PATH.exists():
    raise FileNotFoundError(
        f"Train file not found: {TEST_PATH}"
    )

df_test = pd.read_parquet(TEST_PATH)
VALIDATION_PATH = Path(
    "/home/jetson/Desktop/ciesol-anomaly-detection/data/processed/validation_features_imputed_NaN_before_LSTM.parquet"
)

if not VALIDATION_PATH.exists():
    raise FileNotFoundError(
        f"Train file not found: {VALIDATION_PATH}"
    )

df_validation = pd.read_parquet(VALIDATION_PATH)

print(f"Train dataset loaded: {df_train.shape}")
print(f"Test dataset loaded: {df_test.shape}")
print(f"Validation dataset loaded: {df_validation.shape}")

In [ ]:
# ============================================================
# SCALE DATASETS
# ============================================================

(
    df_train_scaled,
    df_validation_scaled,
    df_test_scaled,
    scaler,
) = scale_train_validation_test(
    train_df=df_train,
    validation_df=df_validation,
    test_df=df_test,
    model_features=MODEL_FEATURES,
)

print("Scaling completed.")

In [ ]:
# ============================================================
# CHECK SCALING FEATURES
# ============================================================

print("=" * 70)
print("SCALING FEATURES")
print("=" * 70)

print(f"\nNumber of continuous/model features: {len(MODEL_FEATURES)}")

for i, feature in enumerate(MODEL_FEATURES, start=1):
    print(f"{i:2d}. {feature}")

scaled_features = [
    f"{feature}_scaled"
    for feature in MODEL_FEATURES
]

print(f"\nNumber of scaled features: {len(scaled_features)}")

print("\nScaled features:")
for col in scaled_features:
    print(" -", col)

print(f"\nBinary missingness feature: {MISSINGNESS_FEATURE}")

print(f"\nTotal LSTM input features: {len(LSTM_FEATURES)}")
print("LSTM features:")
for i, feature in enumerate(LSTM_FEATURES, start=1):
    print(f"{i:2d}. {feature}")

In [ ]:
# ============================================================
# VALIDATE SCALED DATASETS
# ============================================================

from src.scaling import validate_scaled_features

validate_scaled_features(df_train_scaled)
validate_scaled_features(df_validation_scaled)
validate_scaled_features(df_test_scaled)

print("\nAll 19 LSTM input features are valid.")

In [ ]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "scaled"

df_train_scaled.to_parquet(
    PROCESSED_DIR / "train_features_scaled.parquet",
    index=False,
)

df_validation_scaled.to_parquet(
    PROCESSED_DIR / "validation_features_scaled.parquet",
    index=False,
)

df_test_scaled.to_parquet(
    PROCESSED_DIR / "test_features_scaled.parquet",
    index=False,
)

SCALER_PATH = (
    PROJECT_ROOT
    / "data"
    / "scaled"
    / "sensor_feature_scaler.joblib"
)

save_scaler(
    scaler,
    SCALER_PATH,
)

print("Scaled datasets saved successfully.")

In [ ]:
# ============================================================
# INSPECT NaNs BEFORE IMPUTATION
# ============================================================

print("=" * 70)
print("NaN INSPECTION AFTER SCALING")
print("=" * 70)

# ------------------------------------------------------------
# 1. NaNs in the 18 scaled model features
# ------------------------------------------------------------

scaled_features = [
    f"{feature}_scaled"
    for feature in MODEL_FEATURES
]

nan_counts = (
    df_train_scaled[scaled_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nNaN count per scaled feature:")
print(nan_counts)

print(
    f"\nTotal NaN values: "
    f"{nan_counts.sum():,}"
)

# ------------------------------------------------------------
# 2. Rows containing at least one NaN
# ------------------------------------------------------------

rows_with_nan = (
    df_train_scaled[scaled_features]
    .isna()
    .any(axis=1)
)

print(
    f"\nRows containing at least one NaN: "
    f"{rows_with_nan.sum():,}"
)

# ------------------------------------------------------------
# 3. Anomaly type
# ------------------------------------------------------------

print("\nAnomaly type for NaN rows:")

print(
    df_train_scaled.loc[
        rows_with_nan,
        "tipo_anomalia"
    ]
    .value_counts(dropna=False)
)

# ------------------------------------------------------------
# 4. Anomaly source
# ------------------------------------------------------------

print("\nAnomaly source for NaN rows:")

print(
    df_train_scaled.loc[
        rows_with_nan,
        "fuente_anomalia"
    ]
    .value_counts(dropna=False)
)

# ------------------------------------------------------------
# 5. Number of NaN features per row
# ------------------------------------------------------------

nan_per_row = (
    df_train_scaled[scaled_features]
    .isna()
    .sum(axis=1)
)

print("\nNumber of NaN features per affected row:")

print(
    nan_per_row[
        nan_per_row > 0
    ]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 6. Example rows
# ------------------------------------------------------------

print("\nExample rows containing NaNs:")

display(
    df_train_scaled.loc[
        rows_with_nan,
        ["FECHA", "tipo_anomalia", "fuente_anomalia"]
        + MODEL_FEATURES
    ].head(20)
)